# Titanic で機械学習に入門する / ML with the Titanic dataset

**Day 3 の予備モジュール**です。Kaggle(世界最大のデータ分析コンペサイト)で一番有名なデータ「タイタニック号の乗客名簿」を使って、**過去のデータから未来を予測する** 機械学習を体験します。

- 配布ワークシート「Titanic データ分析ワークシート」と一緒に使います
- 1912年の実際の事故のデータです。数字の1つ1つが実在の人だったことを忘れずに

## Step 1 — データを読み込む / Load the data

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
try:
    df = pd.read_csv(url)
except Exception:
    print("ダウンロードできない場合: 先生から titanic.csv をもらって左のフォルダにアップロードしてください")
    df = pd.read_csv("titanic.csv")
print(df.shape)
df.head()

## Step 2 — まず自分の目でデータを見る / Look at the data yourself first

いきなり機械に予測させない。まず**自分の目で**データを見て、生死を分けていそうな「とくちょう」を探そう。

- `Survived`: 1 = 生存, 0 = 死亡 / `Pclass`: 等級(1=富裕 … 3=庶民) / `Sex` / `Age` / `Fare`(運賃)

Before letting a machine predict anything, look at the data with your own eyes. Which columns seem to separate who survived?

In [ ]:
print("全体の生存率 / overall:", round(df["Survived"].mean(), 3))
print("\n性別ごと / by Sex")
print(df.groupby("Sex")["Survived"].mean().round(3))
print("\n等級ごと / by Pclass")
print(df.groupby("Pclass")["Survived"].mean().round(3))

# 年齢を子ども/10代/大人に分けて見る / bin Age into child / teen / adult
df["AgeGroup"] = pd.cut(df["Age"], [0, 12, 18, 200], labels=["child(0-12)", "teen(13-18)", "adult(19+)"])
print("\n年齢グループごと / by Age group")
print(df.groupby("AgeGroup", observed=True)["Survived"].mean().round(3))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
df.groupby("Sex")["Survived"].mean().plot.bar(ax=axes[0], title="by Sex")
df.groupby("Pclass")["Survived"].mean().plot.bar(ax=axes[1], title="by Pclass")
df.groupby("AgeGroup", observed=True)["Survived"].mean().plot.bar(ax=axes[2], title="by Age group")
for ax in axes:
    ax.set_ylim(0, 1)
    ax.set_ylabel("survival rate")
plt.tight_layout()
plt.show()

**観察 / Observe**: どの差が一番大きい? 「女性と子どもを先に救命ボートへ」という当時の行動が、110年後のデータから読み取れる。 / Which gap is biggest? You can read 1912 behaviour straight out of the numbers.

## Step 3 — 予測する前に、予想を書こう / Predict BEFORE you train

機械に答えを出させる前に、**自分の予想**を決める。これが一番の学び。

1. 生死を一番わける「とくちょう」は? **Sex / Pclass / Age / Fare** を重要そうな順に並べる。
2. なぜそう思う? 1行で書く。

Write your ranking down NOW. In a moment the tree will reveal which feature it actually relied on most — then you can check whether your intuition matched the data. **No peeking at Step 4 first!**

## Step 4 — 決定木で予測する / Train a decision tree

**決定木(けっていぎ)** は「質問を繰り返して答えにたどりつく」モデル(20の質問ゲームと同じ)。

(先生向けメモ: 文部科学省「情報Ⅱ」教員研修用教材 第3章 学習15 の R での演習を Python に移したもの)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

data = df[["Pclass", "Sex", "Age", "Fare", "Survived"]].copy()
data["Sex"] = (data["Sex"] == "female").astype(int)  # female=1, male=0
data["Age"] = data["Age"].fillna(data["Age"].median())

X = data[["Pclass", "Sex", "Age", "Fare"]]
y = data["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
print("訓練データの正解率 / train accuracy:", round(tree.score(X_train, y_train), 3))
print("テストデータの正解率 / test accuracy:", round(tree.score(X_test, y_test), 3))

In [ ]:
plt.figure(figsize=(14, 7))
plot_tree(tree, feature_names=["Pclass", "Sex(female=1)", "Age", "Fare"],
          class_names=["died", "survived"], filled=True, fontsize=9)
plt.show()

print("特徴量の重要度 / feature importance — did it match YOUR ranking from Step 3?")
for name, imp in sorted(zip(X.columns, tree.feature_importances_), key=lambda t: -t[1]):
    print(f"  {name:6} {round(imp, 3)}")

**木の読み方 / Read the tree**: 一番上の質問は何? → モデルが「生死を分けた最大の要因」と判断したもの。**あなたの予想(Step 3)と合っていた? / Did the top split match the ranking you wrote?**

## Step 5 — 実験: 木を深くしすぎると? / Overfitting experiment

`max_depth` を変えて、訓練の正解率とテストの正解率がどうなるか見てみよう。

In [ ]:
for depth in [1, 2, 3, 5, 10, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    print(f"max_depth={str(depth):>4} | train {t.score(X_train, y_train):.3f} | test {t.score(X_test, y_test):.3f}")

**問い / Questions**
- 木を深くすると、訓練の正解率はどうなる? テストは?
- 訓練だけ上がってテストが下がる現象を **過学習(丸暗記)** と呼びます。microgpt のラボで見た「図鑑の名前をそのまま言う」のと同じ現象です

## ふりかえり / Reflection

- データから「人の行動」が見えた瞬間はどこ?
- この予測モデルを現実で使うとしたら、どんな危険がある? (バイアスのページとつながる話)
- **Kaggle** に登録すると(13歳以上)、世界中の人と同じ課題に挑戦できます: [kaggle.com/c/titanic](https://www.kaggle.com/c/titanic)